## Topic: Conditional Workflow in LangGraph

### Agenda:
- 1. Introduction of Conditional Workflow

- 2. The Router Function Pattern

- 3. add_conditional_edges() — The Core API

- 4. Key Takeaway

### 1. Introduction of Conditional Workflow
- Definition:
    - A Conditional Workflow is a graph pattern where the next node is chosen dynamically based on the current state. Instead of a fixed path, the graph decides which branch to follow at runtime.
    

- Main idea: 
    - Conditional workflows add decision-making to our graph. The LLM or a function examines the state and routes execution to the appropriate next node.

In [ ]:
""" 
Sequential (Fixed Path):
  A → B → C → END
  Every time: same path. No decisions.

Conditional (Dynamic Path):
       ┌→ B  (if score > 8)
  A →  ├→ C  (if score > 5)
       └→ D  (if score ≤ 5)
  Every time: DIFFERENT path based on state.

"""


""" 
Real-World Analogy: A Hospital Triage:
===================================

  Patient arrives
       │
       ▼
  ┌─────────┐
  │ TRIAGE  │  ← Nurse evaluates severity
  └────┬────┘
       │
  ┌────┼────────┬────────────┐
  ▼    ▼        ▼            ▼
 ICU  Surgery  Outpatient   Pharmacy
(critical) (urgent) (stable) (minor)

  The path depends on the PATIENT'S CONDITION (state),
  not on a fixed schedule.

"""

In [ ]:
""" 
- Why Branching Matters:
===========================

┌─────────────────────────────────────────────────────────────┐
│           WHY BRANCHING MATTERS                             │
│                                                             │
│  1. REAL-WORLD TASKS ARE NOT LINEAR                         │
│     "If the document is relevant, generate an answer.       │
│      If not, search the web instead."                       │
│     A sequential graph cannot do this.                      │
│                                                             │
│  2. AGENTS NEED TO MAKE DECISIONS                           │
│     "Did the LLM call a tool? If yes, execute it.           │
│      If no, return the final answer."                       │
│     This is the core of the ReAct agent pattern.            │
│                                                             │
│  3. QUALITY CONTROL                                         │
│     "Is the answer factually correct? If yes, publish.      │
│      If no, send back for revision."                        │
│     Branching enables self-correction.                      │
│                                                             │
│  4. ERROR HANDLING                                          │
│     "Did the API call succeed? If yes, process data.        │
│      If no, use fallback data."                             │
│                                                             │
│  5. HUMAN-IN-THE-LOOP                                       │
│     "Is the amount over $1000? If yes, ask manager.         │
│      If no, auto-approve."                                  │
└─────────────────────────────────────────────────────────────┘

"""

### 2. The Router Function Pattern

- Definition:
    - A router function is a Python function that reads the current state and returns a string indicating which node to visit next.

### 3. add_conditional_edges() — The Core API


In [ ]:
""" 
- Syntax:
===========

graph.add_conditional_edges(
    source,         # str: The node that routes
    path,           # Callable: The router function
    path_map=None   # dict: Label → destination mapping
)

"""

""" 

┌─────────────────────────────────────────────────────────────┐
│           add_conditional_edges() PARAMETERS                │
│                                                             │
│  source (str):                                              │
│    The node FROM which the conditional edge originates.     │
│    "After this node finishes, call the router."             │
│                                                             │
│  path (Callable):                                           │
│    The router function. Takes state, returns a string.      │
│    def router(state) -> str: ...                            │
│                                                             │
│  path_map (dict, optional):                                 │
│    Maps router return values to destination nodes.          │
│    {"label_a": "node_a", "label_b": "node_b", "end": END}   │
│                                                             │
│    If path_map is None, LangGraph assumes the router        │
│    returns actual node names directly.                      │
└─────────────────────────────────────────────────────────────┘

"""

### 4. Key Takeaway

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│           CONDITIONAL WORKFLOW IN LANGGRAPH                      │
│                                                                  │
│  WHAT:  A graph pattern where the next node is chosen            │
│         dynamically based on the current state.                  │
│                                                                  │
│  PATTERN:                                                        │
│  ────────                                                        │
│       ┌→ Branch A  (if condition X)                              │
│  Node ├→ Branch B  (if condition Y)  → END                       │
│       └→ Branch C  (if condition Z)                              │
│                                                                  │
│  CORE API:                                                       │
│  ─────────                                                       │
│  graph.add_conditional_edges(                                    │
│      "source_node",          # Where the decision happens        │
│      router_function,        # Reads state, returns label        │
│      {"label": "dest_node"}  # Maps labels to destinations       │
│  )                                                               │
│                                                                  │
│  THE ROUTER FUNCTION:                                            │
│  ────────────────────                                            │
│  def router(state: MyState) -> str:                              │
│      if state["condition"]:                                      │
│          return "label_a"    ← Must match a key in path_map      │
│      return "label_b"                                            │
│                                                                  │
│  RULES:                                                          │
│  • Router reads state, returns a string                          │
│  • String must match a key in the path_map                       │
│  • ALL branches must eventually reach END                        │
│  • Don't call LLMs inside routers (do it in a node)              │
│                                                                  │
│  THE MOST IMPORTANT CONDITIONAL PATTERN:                         │
│  ────────────────────────────────────────                        │
│  The ReAct Agent Loop:                                           │
│                                                                  │
│  agent ──┬──→ tools → agent (loop!)                              │
│          └──→ END                                                │
│                                                                  │
│  Router: "Did the LLM call a tool?"                              │
│    Yes → "tools"    (execute tool, then loop back)               │
│    No  → END        (final answer, stop)                         │
│                                                                  │
│  This single pattern is the foundation of ALL AI agents.         │
│                                                                  │
│  COMBINING PATTERNS:                                             │
│  ────────────────────                                            │
│  Real graphs mix all three:                                      │
│  • Sequential: Fixed steps (preprocess → grade)                  │
│  • Parallel: Simultaneous branches (search 3 sources)            │
│  • Conditional: Dynamic routing (relevant? → generate/fallback)  │
│                                                                  │
│  WHEN TO USE:                                                    │
│  ─────────────                                                   │
│  • Agent tool routing (ReAct)                                    │
│  • Quality gates (CRAG, Self-RAG)                                │
│  • Fallback mechanisms                                           │
│  • Classification routing                                        │
│  • Human-in-the-loop decisions                                   │
│                                                                  │
│  WHEN NOT TO USE:                                                │
│  ────────────────                                                │
│  • Path is always the same → SEQUENTIAL                          │
│  • All branches should run → PARALLEL                            │
│  • Logic is trivially simple → Single node if/else               │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Conditional edges are what make agents SMART.                  │
│   Without them, your graph is just a pipeline.                   │
│   With them, your graph can make decisions."                     │
└──────────────────────────────────────────────────────────────────┘

"""